# 04 — Audio embeddings subset (Phase 2, Step 2)

Качаем `embeddings.parquet` (14 GB, 7.72M items) с HF в кэш `huggingface_hub`,
фильтруем по `item_id_to_idx.pkl` из Phase 1 (276,305 items), сохраняем
`[n_items+1, 128]` float32 в `artifacts/audio/embeddings.npy` (~135 MB).

Запускать на **Colab** (95 GB RAM).

In [1]:
# Colab bootstrap (раскомментировать в Colab):
from google.colab import userdata


token = userdata.get('git')
!git clone -b models-1 https://$token@github.com/Vladislavbro/music-recommendations.git
%cd music-recommendations
!pip install -q pyarrow huggingface_hub numpy

Cloning into 'music-recommendations'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 167 (delta 59), reused 145 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 696.75 KiB | 20.49 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/music-recommendations


In [2]:
import sys, pickle
from pathlib import Path


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
# тут надо положить файл вручную в созданную директорию
(Path(PROJECT_ROOT) / 'artifacts' / 'gsasrec').mkdir(parents=True, exist_ok=True)

In [4]:
with open(PROJECT_ROOT / 'artifacts' / 'gsasrec' / 'item_id_to_idx.pkl', 'rb') as f:
    item_id_to_idx = pickle.load(f)
print(f'items: {len(item_id_to_idx):,}, max idx: {max(item_id_to_idx.values()):,}')

items: 276,305, max idx: 276,305


In [5]:
from src.data.audio_embeddings import extract_audio_subset

OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'audio' / 'embeddings.npy'
embeds = extract_audio_subset(item_id_to_idx, OUTPUT_PATH, use_normalized=False)

[audio] downloading embeddings.parquet from HF ...


embeddings.parquet:   0%|          | 0.00/13.8G [00:00<?, ?B/s]

[audio] reading columns ['item_id', 'embed'] ...
[audio] full table: 7,721,749 items, embeds (7721749, 128)
[audio] matched 264,840 / 276,305
[audio] WARNING: 11465 item_idx без эмбеддинга (нули)
[audio] saved (276306, 128) → /content/music-recommendations/artifacts/audio/embeddings.npy (134.9 MB)


In [9]:
# Доп. ячейка в notebooks/04_audio_subset.ipynb — проверка гипотезы "missing = хвост"
import numpy as np, pandas as pd
from src.data.yambda_loader import load_yambda, filter_listens

# 1) поднимаем те же counts, что в Phase 1 (без min_pop фильтра — он бы выкосил часть missing)
data = load_yambda(flavor="50m")
listens = filter_listens(data["interactions"])
counts = listens["item_id"].value_counts()   # pd.Series: item_id -> n_listens
print(f'items в listens: {len(counts):,}, медиана pop: {counts.median():.0f}')

# 2) missing item_ids из нашего npy
arr = np.load(OUTPUT_PATH)
norms = np.linalg.norm(arr[1:], axis=1)
missing_idx = np.where(norms == 0)[0] + 1
idx_to_id = {v: k for k, v in item_id_to_idx.items()}
missing_ids = [idx_to_id[i] for i in missing_idx]
present_ids = [idx_to_id[i] for i in range(1, arr.shape[0]) if i not in set(missing_idx)]

# 3) сравнение распределений popularity
miss_pop = counts.loc[counts.index.intersection(missing_ids)]
pres_pop = counts.loc[counts.index.intersection(present_ids)]

print(f'\nmissing  ({len(miss_pop):,}): median={miss_pop.median():.0f}, '
      f'mean={miss_pop.mean():.0f}, p90={miss_pop.quantile(0.9):.0f}, max={miss_pop.max():.0f}')
print(f'present  ({len(pres_pop):,}): median={pres_pop.median():.0f}, '
      f'mean={pres_pop.mean():.0f}, p90={pres_pop.quantile(0.9):.0f}, max={pres_pop.max():.0f}')

# 4) доля missing на нижней границе
for thr in [5, 10, 50, 100]:
    share = (miss_pop < thr).mean()
    print(f'  доля missing с pop<{thr}: {share:.1%}')


README.md: 0.00B [00:00, ?B/s]

flat/50m/multi_event.parquet:   0%|          | 0.00/384M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47790449 [00:00<?, ? examples/s]

items в listens: 631,003, медиана pop: 3

missing  (11,465): median=15, mean=73, p90=142, max=12770
present  (264,840): median=18, mean=106, p90=195, max=28268
  доля missing с pop<5: 0.0%
  доля missing с pop<10: 33.9%
  доля missing с pop<50: 77.9%
  доля missing с pop<100: 86.8%
